# Klasifikasi DemogPairs Menggunakan ViT (Umur) & Random Forest

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
features = joblib.load('features/demogpairs_vit-age.pkl')
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

Jumlah fitur per gambar: 768


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 20, 30],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf': [1, 2],
        'classifier__max_features': ['sqrt', 'log2'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

RandomForestClassifier: 288 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models, 
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix="models/clf_demogpairs_rf_vit-age_",
    results_path="results/demogpairs_rf_vit-age_"
)
sorted_results = pd.DataFrame(evaluation_results).sort_values(by="test_accuracy", ascending=False).to_dict("records")
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: RandomForestClassifier


{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}


Accuracy  : 0.736574074074074
Precision : 0.7363438845862812
Recall    : 0.736574074074074
F1 Score  : 0.7354170487459699
               precision    recall  f1-score   support

Asian_Females     0.7653    0.7972    0.7810       360
  Asian_Males     0.7380    0.7278    0.7329       360
Black_Females     0.7308    0.6333    0.6786       360
  Black_Males     0.7585    0.8028    0.7800       360
White_Females     0.7180    0.6861    0.7017       360
  White_Males     0.7074    0.7722    0.7384       360

     accuracy                         0.7366      2160
    macro avg     0.7363    0.7366    0.7354      2160
 weighted avg     0.7363    0.7366    0.7354      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9254629629629629,0.7653333333333333,0.7972222222222223,0.7809523809523811,360
Asian_Males,0.9115740740740741,0.7380281690140845,0.7277777777777777,0.7328671328671329,360
Black_Females,0.9,0.7307692307692307,0.6333333333333333,0.6785714285714285,360
Black_Males,0.924537037037037,0.7585301837270341,0.8027777777777778,0.7800269905533063,360
White_Females,0.9027777777777778,0.7180232558139535,0.6861111111111111,0.7017045454545455,360
White_Males,0.9087962962962963,0.7073791348600509,0.7722222222222223,0.7383798140770251,360


Confusion matrix saved: images\cm_rf_vit-age_RandomForestClassifier.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               287                 3                20                30                19                 1
         Asian_Males                 2               262                 3                17                34                42
       Black_Females                41                 7               228                30                13                41
         Black_Males                 9                14                40               289                 4                 4
       White_Females                30                41                 6                 9               247                27
         White_Males                 6                28                15                 6                27               278


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
RandomForestClassifier,models/clf_demogpairs_rf_vit-age_RandomForestClassifier.pkl,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.736574074074074,0.7354170487459699,0.7363438845862812,0.736574074074074,288


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_rf_vit-age_RandomForestClassifier.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 3868.0,
 'days': 0,
 'hours': 1,
 'minutes': 4,
 'seconds': 28.0,
 'text': '0 hari 1 jam 4 menit 28.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 14876.0,
 'days': 0,
 'hours': 4,
 'minutes': 7,
 'seconds': 56.0,
 'text': '0 hari 4 jam 7 menit 56.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.7269,0.7292,0.7182,0.7419,0.7303,0.7293,0.7284,0.7305,0.7293,10.4187
2,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.7269,0.7297,0.7182,0.7373,0.7292,0.7282,0.7273,0.7294,0.7282,10.3294
3,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.7211,0.7245,0.7263,0.739,0.7303,0.7282,0.7271,0.7298,0.7282,9.4279
4,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.7367,0.7141,0.7263,0.7303,0.7326,0.728,0.7265,0.7288,0.728,10.3133
...,...,...,...,...,...,...,...,...,...,...,...
285,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': 'MinMaxScaler'}",0.6464,0.6429,0.6279,0.6389,0.6435,0.6399,0.6371,0.6405,0.6399,6.1626
286,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': 'MinMaxScaler'}",0.6464,0.6429,0.6279,0.6389,0.6435,0.6399,0.637,0.6405,0.6399,5.8902
287,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': None}",0.6464,0.6418,0.6267,0.6389,0.6435,0.6395,0.6366,0.64,0.6395,5.5676
288,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': None}",0.6464,0.6412,0.6267,0.6389,0.6435,0.6394,0.6365,0.6399,0.6394,5.7464
